# 02. Limpieza de BDQueimadas (esqueleto del dataset)

**Propósito.** Limpiar los focos de calor de BDQueimadas (INPE), que en el Hito 3 son el esqueleto del dataset y la base de las variables objetivo. También se calculan los **centroides fijos por municipio** (para mapas y para consultar NASA POWER).

**Entradas** (`BDQueimadas/`): todos los `bdqueimadas_AAAA-*.csv` crudos (2022 a 2026-10-31). Opcional: `DETER/deter_limpio.csv` para verificar coincidencia de municipios.

**Salidas** (`BDQueimadas/`): `bdqueimadas_limpio.csv` y `centroides_municipio.csv`.

**Qué cambió respecto al Hito 2**
- Se leen todos los archivos por patrón (antes eran tres años fijos) y se agregan 2025 y 2026.
- Verificaciones con `assert`: satélite único `AQUA_M-T`, estados de la Amazonía Legal, biomas iguales entre archivos viejos y nuevos, y prueba de regresión de 354,018 focos para 2022 a 2024.
- Se agrega `uf` (sigla de estado) y el archivo de centroides de municipio (antes salían de DETER).
- Ya no se modifica `deter_limpio_merge.csv`: las correcciones de nomenclatura se aplican en memoria.

In [11]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# 1. Entorno: Google Colab (Drive) o ejecucion local
# ---------------------------------------------------------------------------
USAR_DRIVE = True   # True en Colab; False para trabajar con la carpeta local del repositorio
BASE_DRIVE = Path("/content/drive/MyDrive/BIODIVERSITY_GUARD")

if USAR_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        BASE = BASE_DRIVE
    except ImportError:
        print("No se detecto Google Colab: se usa la raiz local del repositorio.")
        USAR_DRIVE = False

if not USAR_DRIVE:
    # la raiz es la primera carpeta (hacia arriba desde la actual) que contiene "datos"
    _aqui = Path.cwd().resolve()
    BASE = next((p for p in [_aqui, *_aqui.parents] if (p / "datos").is_dir()), None)
    if BASE is None:
        raise FileNotFoundError("No se encontro la carpeta 'datos' subiendo desde " + str(_aqui))

if not BASE.exists():
    raise FileNotFoundError(f"No existe BASE = {BASE}. Verifica la carpeta BIODIVERSITY_GUARD en Drive.")

# ---------------------------------------------------------------------------
# 2. Rutas (todas salen de BASE)
# ---------------------------------------------------------------------------
DIR_PREP = BASE / "datos" / "datasets_preprocesamiento"
DIR_BDQ = DIR_PREP / "BDQueimadas"
DIR_DETER = DIR_PREP / "DETER"
DIR_NASA = DIR_PREP / "NasaPower"
DIR_SISAM = DIR_PREP / "SISAM"
DIR_EDA = BASE / "datos" / "datasets_eda"

# ---------------------------------------------------------------------------
# 3. Constantes comunes (identicas en los 7 notebooks)
# ---------------------------------------------------------------------------
FECHA_INICIO_DATOS = "2022-01-01"
FECHA_FIN_EXTRACCION = "2026-11-01"      # DETER y BDQueimadas
PRIMERA_SEMANA = "2022-01-03"             # primer lunes con semana completa (2022-01-01 es sabado)
ULTIMA_SEMANA = "2026-10-19"              # lunes de la ultima semana completa (termina domingo 2026-10-25)
SATELITE_REFERENCIA = "AQUA_M-T"
ESTADOS_AMAZONIA_LEGAL = ["ACRE", "AMAPÁ", "AMAZONAS", "MARANHÃO", "MATO GROSSO", "PARÁ", "RONDÔNIA", "RORAIMA", "TOCANTINS"]
MAPA_UF = {"ACRE": "AC", "AMAPÁ": "AP", "AMAZONAS": "AM", "MARANHÃO": "MA", "MATO GROSSO": "MT", "PARÁ": "PA", "RONDÔNIA": "RO", "RORAIMA": "RR", "TOCANTINS": "TO"}
CORRECCIONES_MUNICIPIO = {"ELDORADO DOS CARAJAS": "ELDORADO DO CARAJAS", "SANTA ISABEL DO PARA": "SANTA IZABEL DO PARA"}
FECHA_CORTE_CENTROIDES = "2024-12-31"     # los centroides se fijan con focos hasta esta fecha

# ---------------------------------------------------------------------------
# 4. Funciones comunes
# ---------------------------------------------------------------------------
def normalizar_texto(texto):
    # mayusculas, sin tildes ni espacios sobrantes
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    return unicodedata.normalize("NFKD", texto).encode("ASCII", "ignore").decode("utf-8")


def semana_lunes(fecha):
    # lunes de la semana (lunes a domingo) de cada fecha; recibe una Serie de fechas
    f = pd.to_datetime(fecha).dt.normalize()
    return f - pd.to_timedelta(f.dt.day_of_week, unit="D")


SEMANAS = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq="W-MON")   # calendario de semanas completas
assert len(SEMANAS) == 251, f"El calendario deberia tener 251 semanas y tiene {len(SEMANAS)}"
assert (SEMANAS.day_of_week == 0).all(), "Todas las semanas deben empezar en lunes"


def verificar_entradas(archivos):
    # imprime los archivos de entrada con su tamano y falla si falta alguno
    faltan = []
    for a in map(Path, archivos):
        if a.exists():
            print(f"  OK  {a.relative_to(BASE)}  ({a.stat().st_size / 1e6:,.1f} MB)")
        else:
            faltan.append(a)
    if faltan:
        lista = "\n".join(f"  - {a}" for a in faltan)
        raise FileNotFoundError("Faltan archivos de entrada (copialos a esas carpetas y vuelve a ejecutar):\n" + lista)


print("BASE:", BASE)
print("Calendario:", SEMANAS[0].date(), "a", SEMANAS[-1].date(), f"({len(SEMANAS)} semanas)")

ESPERADOS_BDQ = [DIR_BDQ / n for n in [
    "bdqueimadas_2022-01-01_2022-12-31.csv", "bdqueimadas_2023-01-01_2023-12-31.csv",
    "bdqueimadas_2024-01-01_2024-12-31.csv", "bdqueimadas_2025-01-01_2025-12-31.csv",
    "bdqueimadas_2026-01-01_2026-10-31.csv"]]
print("\nArchivos de entrada esperados:")
verificar_entradas(ESPERADOS_BDQ)
FOCOS_2022_2024_HITO2 = 354018   # focos del bdqueimadas_limpio.csv del Hito 2 (prueba de regresion)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BASE: /content/drive/MyDrive/BIODIVERSITY_GUARD
Calendario: 2022-01-03 a 2026-10-19 (251 semanas)

Archivos de entrada esperados:
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_2022-01-01_2022-12-31.csv  (11.8 MB)
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_2023-01-01_2023-12-31.csv  (10.0 MB)
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_2024-01-01_2024-12-31.csv  (14.4 MB)
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_2025-01-01_2025-12-31.csv  (4.4 MB)
  OK  datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_2026-01-01_2026-10-31.csv  (4.8 MB)


## Lectura de todos los archivos

Se leen con un patrón (`bdqueimadas_AAAA-*.csv`), no con una lista fija. Los archivos de 2025 y 2026 traen un carácter BOM (`﻿`) incrustado en medio del archivo, pegado a la `DataHora` de una fila; si no se elimina, esa fila se pierde al convertir la fecha. Además, en esos archivos `DiaSemChuva`, `Precipitacao` y `RiscoFogo` pueden venir **vacíos** (en 2022 a 2024 venían con `-999`); ambos casos terminan como valor nulo.

In [12]:
def leer_queimadas(ruta):
    d = pd.read_csv(ruta, encoding="utf-8-sig")
    d["DataHora"] = d["DataHora"].astype(str)
    n_bom = d["DataHora"].str.contains("﻿", regex=False).sum()
    if n_bom:
        print(f"  {ruta.name}: {n_bom} fila(s) con BOM incrustado en DataHora (se elimina)")
    d["DataHora"] = d["DataHora"].str.replace("﻿", "", regex=False).str.strip()
    d["anio_archivo"] = int(ruta.name.split("_")[1][:4])
    return d


archivos = sorted(DIR_BDQ.glob("bdqueimadas_[0-9]*.csv"))
print("Archivos leidos:", [a.name for a in archivos])
assert set(ESPERADOS_BDQ) <= set(archivos), "Faltan archivos esperados de BDQueimadas"

COLUMNAS_CRUDAS = ["DataHora", "Satelite", "Pais", "Estado", "Municipio", "Bioma",
                   "DiaSemChuva", "Precipitacao", "RiscoFogo", "FRP", "Latitude", "Longitude"]
partes = []
for a in archivos:
    d = leer_queimadas(a)
    assert d.columns.drop("anio_archivo").tolist() == COLUMNAS_CRUDAS, f"Cabecera inesperada en {a.name}: {d.columns.tolist()}"
    partes.append(d)
    print(f"  {a.name}: {len(d):,} filas")

df_queimadas = pd.concat(partes, ignore_index=True)
for c in ["DiaSemChuva", "Precipitacao", "RiscoFogo", "FRP", "Latitude", "Longitude"]:
    antes = df_queimadas[c].notna().sum()
    df_queimadas[c] = pd.to_numeric(df_queimadas[c], errors="coerce")
    assert df_queimadas[c].notna().sum() == antes, f"La columna {c} tiene valores no numericos"
print("\nTotal combinado:", df_queimadas.shape)

Archivos leidos: ['bdqueimadas_2022-01-01_2022-12-31.csv', 'bdqueimadas_2023-01-01_2023-12-31.csv', 'bdqueimadas_2024-01-01_2024-12-31.csv', 'bdqueimadas_2025-01-01_2025-12-31.csv', 'bdqueimadas_2026-01-01_2026-10-31.csv']
  bdqueimadas_2022-01-01_2022-12-31.csv: 115,033 filas
  bdqueimadas_2023-01-01_2023-12-31.csv: 98,639 filas
  bdqueimadas_2024-01-01_2024-12-31.csv: 140,346 filas
  bdqueimadas_2025-01-01_2025-12-31.csv: 1 fila(s) con BOM incrustado en DataHora (se elimina)
  bdqueimadas_2025-01-01_2025-12-31.csv: 43,131 filas
  bdqueimadas_2026-01-01_2026-10-31.csv: 1 fila(s) con BOM incrustado en DataHora (se elimina)
  bdqueimadas_2026-01-01_2026-10-31.csv: 48,487 filas

Total combinado: (445636, 13)


## Verificaciones de los archivos crudos

- (a) `Satelite` contiene únicamente `AQUA_M-T` (satélite de referencia, para que la serie sea comparable entre años).
- (b) Los estados pertenecen a la Amazonía Legal.
- (c) Por año: conteo de focos, biomas y estados; y los biomas de los archivos nuevos deben ser los mismos que los de 2022 a 2024 (no se fuerza el valor del bioma).

In [13]:
assert set(df_queimadas["Satelite"].unique()) == {SATELITE_REFERENCIA}, \
    f"Satelites inesperados: {df_queimadas['Satelite'].unique()}"
assert set(df_queimadas["Estado"].unique()) <= set(ESTADOS_AMAZONIA_LEGAL), \
    f"Estados fuera de la Amazonia Legal: {set(df_queimadas['Estado'].unique()) - set(ESTADOS_AMAZONIA_LEGAL)}"

resumen = df_queimadas.groupby("anio_archivo").agg(
    focos=("FRP", "size"),
    biomas=("Bioma", lambda s: ", ".join(sorted(s.unique()))),
    n_estados=("Estado", "nunique"),
)
print(resumen)

biomas_ref = set(df_queimadas.loc[df_queimadas["anio_archivo"] <= 2024, "Bioma"].unique())
print("\nBiomas de los archivos 2022 a 2024:", sorted(biomas_ref))
for anio in sorted(df_queimadas.loc[df_queimadas["anio_archivo"] > 2024, "anio_archivo"].unique()):
    b = set(df_queimadas.loc[df_queimadas["anio_archivo"] == anio, "Bioma"].unique())
    assert b == biomas_ref, f"Los biomas de {anio} ({sorted(b)}) difieren de los de 2022 a 2024 ({sorted(biomas_ref)})"
print("OK: satelite unico, estados de la Amazonia Legal y biomas consistentes.")

               focos    biomas  n_estados
anio_archivo                             
2022          115033  Amazônia          9
2023           98639  Amazônia          9
2024          140346  Amazônia          9
2025           43131  Amazônia          9
2026           48487  Amazônia          9

Biomas de los archivos 2022 a 2024: ['Amazônia']
OK: satelite unico, estados de la Amazonia Legal y biomas consistentes.


In [14]:
# fecha y hora (formato AAAA/MM/DD HH:MM:SS)
df_queimadas["DataHora"] = pd.to_datetime(df_queimadas["DataHora"], format="%Y/%m/%d %H:%M:%S", errors="coerce")
assert df_queimadas["DataHora"].notna().all(), f"{df_queimadas['DataHora'].isna().sum()} fechas invalidas tras la conversion"

df_queimadas["fecha"] = df_queimadas["DataHora"].dt.normalize()
df_queimadas["hora"] = df_queimadas["DataHora"].dt.hour

assert df_queimadas["fecha"].min() >= pd.Timestamp(FECHA_INICIO_DATOS)
assert df_queimadas["fecha"].max() < pd.Timestamp(FECHA_FIN_EXTRACCION)
assert (df_queimadas["fecha"].dt.year == df_queimadas["anio_archivo"]).all(), "Hay focos en un archivo de otro anio"
print("Rango de fechas:", df_queimadas["fecha"].min().date(), "a", df_queimadas["fecha"].max().date())
print("Focos por anio:\n", df_queimadas["fecha"].dt.year.value_counts().sort_index())

Rango de fechas: 2022-01-01 a 2026-10-31
Focos por anio:
 fecha
2022    115033
2023     98639
2024    140346
2025     43131
2026     48487
Name: count, dtype: int64


### Hallazgos de esta etapa

**Qué se observa**
- Hay **445,636** focos de calor en cinco archivos. Por año: **115,033** (2022), **98,639** (2023), **140,346** (2024), **43,131** (2025) y **48,487** (2026).
- Todos los focos son del satélite `AQUA_M-T`, del bioma Amazônia y de **9 estados** de la Amazonía Legal.
- Los archivos de 2025 y 2026 traen **1 fila cada uno** con un carácter BOM incrustado en `DataHora`.

**Por qué se decidió así**
- Un satélite único (`AQUA_M-T`) hace que la serie sea comparable entre años.
- El BOM se elimina antes de convertir la fecha. Si no, esa fila se perdería.

**Qué implica para el modelado**
- El nivel de 2025 y 2026 es muy distinto al de 2022 a 2024. Hay que tenerlo en cuenta al dividir entre entrenamiento y prueba.


## Valores faltantes (`-999` y vacíos)

El código `-999` de los archivos 2022 a 2024 se convierte a nulo, igual que en el Hito 2. En 2025 y 2026 los faltantes vienen como celdas vacías. **Se informa el porcentaje de nulos por año** de `RiscoFogo`, porque esa variable alimenta el target de clasificación; no se imputa aquí.

In [15]:
COLS_CLIMA = ["DiaSemChuva", "Precipitacao", "RiscoFogo"]
print("Filas con -999 antes de convertir:\n", (df_queimadas[COLS_CLIMA] == -999).sum())
df_queimadas[COLS_CLIMA] = df_queimadas[COLS_CLIMA].replace(-999, np.nan)

nulos = df_queimadas.groupby(df_queimadas["fecha"].dt.year)[COLS_CLIMA + ["FRP"]].apply(lambda g: g.isna().mean() * 100).round(2)
nulos.index.name = "anio"
print("\nPorcentaje de nulos por anio:")
print(nulos)
print("\nDescribe despues de limpiar:")
print(df_queimadas[COLS_CLIMA + ["FRP"]].describe())

Filas con -999 antes de convertir:
 DiaSemChuva     7625
Precipitacao       0
RiscoFogo       1496
dtype: int64

Porcentaje de nulos por anio:
      DiaSemChuva  Precipitacao  RiscoFogo  FRP
anio                                           
2022         1.72          0.00       0.25  0.0
2023         1.74          0.00       0.42  0.0
2024         1.40          0.00       0.37  0.0
2025         9.20          6.37       6.74  0.0
2026        11.47          9.94      10.18  0.0

Describe despues de limpiar:
         DiaSemChuva   Precipitacao      RiscoFogo            FRP
count  430440.000000  438065.000000  436569.000000  445636.000000
mean       13.573230       0.956011       0.609183      56.962098
std        23.166354       4.034418       0.369278     135.024021
min         0.000000       0.000000       0.000000       0.000000
25%         2.000000       0.000000       0.250000      13.400000
50%         5.000000       0.000000       0.700000      24.800000
75%        13.000000       0.

### Hallazgos de esta etapa

**Qué se observa**
- El código `-999` aparecía en **7,625** filas de `DiaSemChuva` y **1,496** de `RiscoFogo`. Se convirtió a nulo.
- Los nulos suben en 2025 y 2026:

| Año | `DiaSemChuva` | `Precipitacao` | `RiscoFogo` |
|---|---|---|---|
| 2022 | 1.72 % | 0.00 % | 0.25 % |
| 2023 | 1.74 % | 0.00 % | 0.42 % |
| 2024 | 1.40 % | 0.00 % | 0.37 % |
| 2025 | 9.20 % | 6.37 % | 6.74 % |
| 2026 | 11.47 % | 9.94 % | 10.18 % |

**Por qué se decidió así**
- No se imputa en esta etapa: aquí solo se limpia y se informa el porcentaje por año.

**Qué implica para el modelado**
- **Limitación:** `RiscoFogo` alimenta el target de clasificación y pasa de menos de **0.5 %** de nulos en 2022 a 2024 a **6.74 %** en 2025 y **10.18 %** en 2026. Hay que declararlo en el informe.


In [16]:
df_queimadas["municipio_norm"] = df_queimadas["Municipio"].apply(normalizar_texto)
df_queimadas["uf"] = df_queimadas["Estado"].map(MAPA_UF)
assert df_queimadas["uf"].notna().all(), "Hay estados sin mapear a sigla"

print("Municipios unicos (nombre):", df_queimadas["municipio_norm"].nunique())
print("Pares unicos (municipio, uf):", df_queimadas[["municipio_norm", "uf"]].drop_duplicates().shape[0])
print("Filas 100% duplicadas (informativo, no se eliminan):", df_queimadas.duplicated().sum())

Municipios unicos (nombre): 539
Pares unicos (municipio, uf): 543
Filas 100% duplicadas (informativo, no se eliminan): 0


## Prueba de regresión

El `bdqueimadas_limpio.csv` del Hito 2 tenía **354,018** focos (2022 a 2024). Los archivos crudos de esos años no cambiaron, así que el conteo debe ser idéntico. Si no lo es, algo se rompió en la lectura o la limpieza.

In [17]:
n_2022_2024 = int((df_queimadas["fecha"].dt.year <= 2024).sum())
assert n_2022_2024 == FOCOS_2022_2024_HITO2, \
    f"Regresion fallida: se esperaban {FOCOS_2022_2024_HITO2:,} focos en 2022 a 2024 y hay {n_2022_2024:,}"
print(f"OK: {n_2022_2024:,} focos en 2022 a 2024 (igual al Hito 2)")

OK: 354,018 focos en 2022 a 2024 (igual al Hito 2)


## Centroides fijos por municipio

**Decisión:** el centroide de cada `(municipio_norm, uf)` es el promedio de `Latitude` y `Longitude` de los focos con fecha hasta 2024-12-31. Así es fijo y no depende del periodo de prueba (no hay fuga de información). Si un municipio solo tiene focos en 2025 o 2026, se usa el promedio de todos sus focos y se marca `centroide_fuente = "2025_2026"`; en los demás casos `"2022_2024"`.

In [18]:
CLAVE = ["municipio_norm", "uf"]


def calcular_centroides(focos):
    agg = dict(lat=("Latitude", "mean"), lon=("Longitude", "mean"), n_focos=("Latitude", "size"))
    ref = focos[focos["fecha"] <= pd.Timestamp(FECHA_CORTE_CENTROIDES)]
    c_ref = ref.groupby(CLAVE).agg(**agg).reset_index()
    c_ref["centroide_fuente"] = "2022_2024"

    pares_ref = set(map(tuple, c_ref[CLAVE].values))
    resto = focos[[tuple(p) not in pares_ref for p in focos[CLAVE].values]]
    c_resto = resto.groupby(CLAVE).agg(**agg).reset_index()
    c_resto["centroide_fuente"] = "2025_2026"
    return pd.concat([c_ref, c_resto], ignore_index=True).sort_values(CLAVE).reset_index(drop=True)


centroides = calcular_centroides(df_queimadas)
assert not centroides.duplicated(CLAVE).any()
assert centroides[["lat", "lon"]].notna().all().all()
assert centroides["lat"].between(-34, 6).all() and centroides["lon"].between(-75, -33).all()

print("Municipios con centroide:", len(centroides))
print(centroides["centroide_fuente"].value_counts())
centroides.to_csv(DIR_BDQ / "centroides_municipio.csv", index=False, encoding="utf-8")
print("Guardado:", (DIR_BDQ / "centroides_municipio.csv").relative_to(BASE))
centroides.head()

Municipios con centroide: 543
centroide_fuente
2022_2024    542
2025_2026      1
Name: count, dtype: int64
Guardado: datos/datasets_preprocesamiento/BDQueimadas/centroides_municipio.csv


,municipio_norm,uf,lat,lon,n_focos,centroide_fuente
0,ABAETETUBA,PA,-1.773752,-48.785494,240,2022_2024
1,ABEL FIGUEIREDO,PA,-4.931251,-48.432482,115,2022_2024
2,ACAILANDIA,MA,-4.794587,-47.423763,381,2022_2024
3,ACARA,PA,-2.057389,-48.384678,1856,2022_2024
4,ACRELANDIA,AC,-9.938650,-66.902844,391,2022_2024


### Hallazgos de esta etapa

**Qué se observa**
- La prueba de regresión pasó: **354,018** focos en 2022 a 2024, igual que en el Hito 2.
- Hay **539** nombres de municipio únicos pero **543** pares `(municipio, uf)`. Se calcularon **543** centroides: **542** con focos de 2022 a 2024 y **1** con focos de 2025 y 2026.

**Por qué se decidió así**
- La clave es siempre el par `(municipio_norm, uf)`, porque un mismo nombre aparece en más de un estado.
- El centroide es fijo y sale de focos hasta 2024-12-31, para que no dependa del periodo de prueba.


## Coincidencia de municipios con DETER

DETER y BDQueimadas nombran distinto a dos municipios (`ELDORADO DOS CARAJAS` y `SANTA ISABEL DO PARA`). Las correcciones se aplican **en memoria** sobre DETER (ya no se escribe `deter_limpio_merge.csv`). La comparación usa el par `(municipio_norm, uf)`, nunca solo el nombre (hay homónimos, por ejemplo Pau D'Arco en PA y TO).

In [19]:
ruta_deter = DIR_DETER / "deter_limpio.csv"
if not ruta_deter.exists():
    print("Aun no existe DETER/deter_limpio.csv (ejecutar antes el notebook 01): se omite la comparacion.")
else:
    deter = pd.read_csv(ruta_deter, usecols=["municipio_norm", "uf"]).drop_duplicates()
    deter["municipio_norm"] = deter["municipio_norm"].replace(CORRECCIONES_MUNICIPIO)
    p_deter = set(map(tuple, deter[CLAVE].drop_duplicates().values))
    p_focos = set(map(tuple, centroides[CLAVE].values))
    print(f"Pares en DETER: {len(p_deter)} | en focos: {len(p_focos)} | en ambos: {len(p_deter & p_focos)}")
    print("Solo en DETER (sin focos):", sorted(p_deter - p_focos))
    print("Solo en focos (sin alertas DETER):", len(p_focos - p_deter))

Pares en DETER: 480 | en focos: 543 | en ambos: 471
Solo en DETER (sin focos): [('AUGUSTINOPOLIS', 'TO'), ('BARAO DE MELGACO', 'MT'), ('CAMPO NOVO DO PARECIS', 'MT'), ('NOSSA SENHORA DO LIVRAMENTO', 'MT'), ('NOVO SANTO ANTONIO', 'MT'), ('SAMPAIO', 'TO'), ('SANTO ANTONIO DO LEVERGER', 'MT'), ('SAPEZAL', 'MT'), ('SERRA NOVA DOURADA', 'MT')]
Solo en focos (sin alertas DETER): 72


In [20]:
columnas_finales = ["fecha", "hora", "municipio_norm", "Estado", "uf", "DiaSemChuva", "Precipitacao",
                    "RiscoFogo", "FRP", "Latitude", "Longitude"]
df_limpio = df_queimadas[columnas_finales].copy()

df_limpio.to_csv(DIR_BDQ / "bdqueimadas_limpio.csv", index=False, encoding="utf-8")
print("Guardado:", (DIR_BDQ / "bdqueimadas_limpio.csv").relative_to(BASE), df_limpio.shape)
print(df_limpio.dtypes)

Guardado: datos/datasets_preprocesamiento/BDQueimadas/bdqueimadas_limpio.csv (445636, 11)
fecha             datetime64[ns]
hora                       int32
municipio_norm            object
Estado                    object
uf                        object
DiaSemChuva              float64
Precipitacao             float64
RiscoFogo                float64
FRP                      float64
Latitude                 float64
Longitude                float64
dtype: object
